In [18]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np

# Show all columns when printing
pd.set_option('display.max_columns', None)
pd.set_option('display.max_rows', None)

In [19]:
features = pd.read_csv("src/data/training/processed/final_training_data.csv")

In [20]:
features['mix__sd_od_history_to_kyc_ratio'] = features['od__history_length_days'] / features['sd__age_since_first_kyc']

In [21]:
features = features.drop(columns=['user_id', 'reference_date', 'default_date'])
features = features.loc[:, ~features.columns.str.startswith('dn__')]

In [23]:
from itertools import combinations
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt

# ============================================================
# CONFIG
# ============================================================

feature_groups = ['ab__', 'cb__', 'sd__', 'ep__', 'madrid__',
                  'mv__', 'ob__', 'ul__', 'mix__']

corr_threshold = 0.0


# ============================================================
# PREPARE DATA
# ============================================================

# Collect all block features
all_feature_cols = [
    c for c in features.columns
    if any(c.startswith(prefix) for prefix in feature_groups)
]

df_all = features[all_feature_cols]
df_numeric = (
    df_all
    .select_dtypes(include=np.number)
    .replace([np.inf, -np.inf], np.nan)
    .dropna()
)

numeric_cols = df_numeric.columns
pearson_corr = df_numeric.corr(method="pearson")

print("=" * 100)
print("MULTIVARIATE CORRELATION ANALYSIS")
print("=" * 100)


# ============================================================
# HELPER: Identify Block
# ============================================================

def get_block(col):
    return next((p for p in feature_groups if col.startswith(p)), "unknown")


# ============================================================
# 1️⃣ WITHIN-GROUP ANALYSIS
# ============================================================

print("\n" + "="*100)
print("1️⃣ WITHIN-GROUP HIGH CORRELATION")
print("="*100)

within_pairs = []

for prefix in feature_groups:

    block_cols = [c for c in numeric_cols if c.startswith(prefix)]

    if len(block_cols) < 2:
        continue

    for col1, col2 in combinations(block_cols, 2):

        corr_val = pearson_corr.loc[col1, col2]

        if abs(corr_val) > corr_threshold:
            within_pairs.append(
                (col1, col2, prefix, prefix, corr_val)
            )

within_df = pd.DataFrame(
    within_pairs,
    columns=["Feature_1", "Feature_2", "Block_1", "Block_2", "Pearson_Corr"]
).sort_values(by="Pearson_Corr", key=abs, ascending=False)

display(within_df)


# ============================================================
# 2️⃣ CROSS-GROUP ANALYSIS (BLOCK vs REST)
# ============================================================

print("\n" + "="*100)
print("2️⃣ CROSS-GROUP HIGH CORRELATION (BLOCK vs REST)")
print("="*100)

cross_pairs = []

for prefix in feature_groups:

    block_cols = [c for c in numeric_cols if c.startswith(prefix)]
    other_cols = [c for c in numeric_cols if not c.startswith(prefix)]

    for col1 in block_cols:
        for col2 in other_cols:

            corr_val = pearson_corr.loc[col1, col2]

            if abs(corr_val) > corr_threshold:

                cross_pairs.append(
                    (col1, col2, prefix, get_block(col2), corr_val)
                )

cross_df = pd.DataFrame(
    cross_pairs,
    columns=["Feature_1", "Feature_2", "Block_1", "Block_2", "Pearson_Corr"]
).sort_values(by="Pearson_Corr", key=abs, ascending=False)

display(cross_df)

MULTIVARIATE CORRELATION ANALYSIS

1️⃣ WITHIN-GROUP HIGH CORRELATION


,Feature_1,Feature_2,Block_1,Block_2,Pearson_Corr
425,ob__num_od_exits_6m,ob__full_repayment_events_6m,ob__,ob__,1.000000
505,ob__technical_od_days_6m,ob__technical_od_deep_days_6m,ob__,ob__,1.000000
482,ob__od_draw_days_6m,ob__od_enabled_zero_util_days_6m,ob__,ob__,-0.995137
407,ob__num_od_entries_6m,ob__num_od_exits_6m,ob__,ob__,0.994668
409,ob__num_od_entries_6m,ob__full_repayment_events_6m,ob__,ob__,0.994668
264,ob__avg_util_change_3m,ob__util_slope_3m,ob__,ob__,0.955186
109,madrid__avg_monthly_expenses_3m,madrid__avg_monthly_repayment_capacity_3m,madrid__,madrid__,0.945111
74,ab__bal_change_freq_3m,ab__days_down_6m,ab__,ab__,0.936667
125,ob__limit_ref,ob__balance_ref,ob__,ob__,0.912377
331,ob__std_util_3m,ob__util_range_3m,ob__,ob__,0.905077



2️⃣ CROSS-GROUP HIGH CORRELATION (BLOCK vs REST)


,Feature_1,Feature_2,Block_1,Block_2,Pearson_Corr
1816,ob__balance_ref,ab__avg_bal_3m,ob__,ab__,-0.850842
1,ab__avg_bal_3m,ob__balance_ref,ab__,ob__,-0.850842
0,ab__avg_bal_3m,ob__limit_ref,ab__,ob__,-0.779763
1782,ob__limit_ref,ab__avg_bal_3m,ob__,ab__,-0.779763
418,ab__mean_abs_change_6m,madrid__avg_monthly_repayment_capacity_3m,ab__,madrid__,0.757131
1408,madrid__avg_monthly_repayment_capacity_3m,ab__mean_abs_change_6m,madrid__,ab__,0.757131
1347,madrid__avg_monthly_expenses_3m,ab__mean_abs_change_6m,madrid__,ab__,0.706810
417,ab__mean_abs_change_6m,madrid__avg_monthly_expenses_3m,ab__,madrid__,0.706810
1409,madrid__avg_monthly_repayment_capacity_3m,ab__std_abs_change_6m,madrid__,ab__,0.637517
468,ab__std_abs_change_6m,madrid__avg_monthly_repayment_capacity_3m,ab__,madrid__,0.637517
